In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/ttc-bus-delay-data/ttc-bus-delay-data-2024-clean.csv")

df.show(5)

+-------------------+-----+-------------------+------+--------------------+-------------+---------+-------+---------+-------+
|               Date|Route|               Time|   Day|            Location|     Incident|Min Delay|Min Gap|Direction|Vehicle|
+-------------------+-----+-------------------+------+--------------------+-------------+---------+-------+---------+-------+
|2024-01-01 00:00:00|   89|2026-08-02 02:08:00|Monday|  KEELE AND GLENLAKE|       Vision|       10|     20|        N|   7107|
|2024-01-01 00:00:00|   39|2026-08-02 02:30:00|Monday|       FINCH STATION|General Delay|       20|     40|      NaN|   8914|
|2024-01-01 00:00:00|  300|2026-08-02 03:13:00|Monday|   BLOOR AND MANNING|General Delay|        0|      0|      NaN|   8562|
|2024-01-01 00:00:00|   65|2026-08-02 03:23:00|Monday|PARLIAMENT AND BLOOR|     Security|        0|      0|        N|   8574|
|2024-01-01 00:00:00|  113|2026-08-02 03:37:00|Monday|        MAIN STATION|     Security|        0|      0|      NaN| 

In [0]:
from pyspark.sql.functions import date_format

df = df.withColumn("Time", date_format("Time", "HH:mm:ss"))
df.show(5)

+-------------------+-----+--------+------+--------------------+-------------+---------+-------+---------+-------+
|               Date|Route|    Time|   Day|            Location|     Incident|Min Delay|Min Gap|Direction|Vehicle|
+-------------------+-----+--------+------+--------------------+-------------+---------+-------+---------+-------+
|2024-01-01 00:00:00|   89|02:08:00|Monday|  KEELE AND GLENLAKE|       Vision|       10|     20|        N|   7107|
|2024-01-01 00:00:00|   39|02:30:00|Monday|       FINCH STATION|General Delay|       20|     40|      NaN|   8914|
|2024-01-01 00:00:00|  300|03:13:00|Monday|   BLOOR AND MANNING|General Delay|        0|      0|      NaN|   8562|
|2024-01-01 00:00:00|   65|03:23:00|Monday|PARLIAMENT AND BLOOR|     Security|        0|      0|        N|   8574|
|2024-01-01 00:00:00|  113|03:37:00|Monday|        MAIN STATION|     Security|        0|      0|      NaN|   8541|
+-------------------+-----+--------+------+--------------------+-------------+--

In [0]:
from pyspark.sql.functions import avg, count, round

route_delays = df.groupBy("Route") \
    .agg(
        avg("Min Delay").alias("avg_delay"),
        count("*").alias("num_incidents")
    ) \
    .orderBy(round("avg_delay", 2).desc())

route_delays.show(10)

+-----+------------------+-------------+
|Route|         avg_delay|num_incidents|
+-----+------------------+-------------+
|   77|136.17741935483872|           62|
|  340|             117.0|            2|
|  202| 97.33870967741936|           62|
|   28| 91.28571428571429|           35|
|  121| 71.88127853881278|          219|
|   55|         65.171875|           64|
|   YU| 64.28571428571429|            7|
|  354| 64.28571428571429|           21|
|   93| 61.97727272727273|           44|
|  162|58.382716049382715|           81|
+-----+------------------+-------------+
only showing top 10 rows


In [0]:
reliable_route_delays = route_delays.filter(route_delays.num_incidents >= 20)
reliable_route_delays.show(10)

+-----+------------------+-------------+
|Route|         avg_delay|num_incidents|
+-----+------------------+-------------+
|   77|136.17741935483872|           62|
|  202| 97.33870967741936|           62|
|   28| 91.28571428571429|           35|
|  121| 71.88127853881278|          219|
|   55|         65.171875|           64|
|  354| 64.28571428571429|           21|
|   93| 61.97727272727273|           44|
|  162|58.382716049382715|           81|
|  106| 54.32085561497326|          187|
|   33| 53.19444444444444|           36|
+-----+------------------+-------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import hour

# Extract just the hour from the Time column
df_with_hour = df.withColumn("hour_of_day", hour("Time"))

hourly_delays = df_with_hour.groupBy("hour_of_day") \
    .agg(
        avg("Min Delay").alias("avg_delay"),
        count("*").alias("num_incidents")
    ) \
    .orderBy("hour_of_day")

hourly_delays.show(24)

+-----------+------------------+-------------+
|hour_of_day|         avg_delay|num_incidents|
+-----------+------------------+-------------+
|          0|19.540016849199663|         1187|
|          1| 19.75664187035069|          941|
|          2|25.859782608695653|          920|
|          3|22.787337662337663|          616|
|          4|17.742441209406493|          893|
|          5| 19.56437054631829|         2105|
|          6|25.239923954372625|         2630|
|          7| 40.55862329803328|         2644|
|          8|25.116270145817346|         2606|
|          9| 23.16151202749141|         2619|
|         10| 20.85354691075515|         2622|
|         11|21.396442687747037|         2530|
|         12|19.465432565168115|         2647|
|         13|18.112191958495462|         3084|
|         14|19.344853875476492|         3935|
|         15|19.115040075436116|         4242|
|         16|15.899930183849197|         4297|
|         17| 19.75254730713246|         4122|
|         18|

In [0]:
incident_delays = df.groupBy("Incident") \
    .agg(
        avg("Min Delay").alias("avg_delay"),
        count("*").alias("num_incidents")
    ) \
    .filter("num_incidents >= 20") \
    .orderBy(avg("Min Delay").desc())

incident_delays.show(20)

+--------------------+------------------+-------------+
|            Incident|         avg_delay|num_incidents|
+--------------------+------------------+-------------+
|           Diversion|118.80331541218638|         4464|
|Road Blocked - NO...| 34.78688524590164|          183|
|Operations - Oper...|14.843625893566323|        10072|
|Cleaning - Unsani...|  14.7380042462845|         2355|
|              Vision|14.111053450960041|         1927|
|  Utilized Off Route| 13.63914373088685|         2616|
|          Mechanical| 13.61190044336961|        19848|
|       Investigation|12.967156439066551|         1157|
|     Collision - TTC| 12.62845469838981|         4161|
|            Security|12.002275077559462|         4835|
|  Emergency Services|11.684035140866404|         3301|
|       General Delay| 11.02486591906387|         4102|
+--------------------+------------------+-------------+



In [0]:
day_delays = df.groupBy("Day") \
    .agg(
        avg("Min Delay").alias("avg_delay"),
        count("*").alias("num_incidents")
    ) \
    .orderBy(avg("Min Delay").desc())

day_delays.show(7)

+---------+------------------+-------------+
|      Day|         avg_delay|num_incidents|
+---------+------------------+-------------+
|   Monday|22.752089831565815|         8015|
|Wednesday| 22.07973797153829|         8854|
|   Sunday| 21.71838056680162|         6175|
|  Tuesday|21.379928315412187|         9207|
| Thursday| 21.32597604259095|         9016|
|   Friday| 20.79330667227952|         9502|
| Saturday|19.955283567619972|         8252|
+---------+------------------+-------------+



In [0]:
location_delays = df.groupBy("Location") \
    .agg(
        avg("Min Delay").alias("avg_delay"),
        count("*").alias("num_incidents")
    ) \
    .filter("num_incidents >= 20") \
    .orderBy(avg("Min Delay").desc())

location_delays.show(10)

+--------------------+-----------------+-------------+
|            Location|        avg_delay|num_incidents|
+--------------------+-----------------+-------------+
|    JANE AND STEELES|            135.5|           58|
|     KEELE AND FINCH|         123.5625|           32|
|     PIONEER VILLAGE|            117.0|           29|
|    STEELES AND JANE| 97.3103448275862|           29|
| WESTON AND EGLINTON|95.61538461538461|           39|
|   YONGE AND GERRARD|             95.0|           20|
|PROGRESS AND SHEP...|82.72222222222223|           36|
|  YONGE AND EGLINTON|77.37931034482759|           29|
|LIBERTY AND ATLANTIC|74.16666666666667|           30|
|   AVENUE AND WILSON|69.81818181818181|           22|
+--------------------+-----------------+-------------+
only showing top 10 rows


In [0]:
route_delays.write.mode("overwrite").saveAsTable("workspace.default.route_delays")
hourly_delays.write.mode("overwrite").saveAsTable("workspace.default.hourly_delays")
incident_delays.write.mode("overwrite").saveAsTable("workspace.default.incident_delays")
day_delays.write.mode("overwrite").saveAsTable("workspace.default.day_delays")
location_delays.write.mode("overwrite").saveAsTable("workspace.default.location_delays")

print("All five summary tables saved!")

All five summary tables saved!
